# Setup
## Dependencies

In [ ]:
%config InlineBackend.figure_format = 'retina'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re
from scipy.optimize import curve_fit
from astropy.cosmology import FlatLambdaCDM

## File paths

In [ ]:
# Location of SNooPy output files
data_dir = Path("../output snoopy")

# Find all supernova summary files
summary_files = sorted(data_dir.glob("*_summary.txt"))


for file in summary_files:
    print(file.name)

## Extracting the data 

In [ ]:
def extract_sn_data(file):

    # Read the summary file
    text = file.read_text(encoding="utf-8")

    # Supernova name from filename
    name = file.stem.replace("_summary", "")

    # Extract CMB-frame redshift
    z_match = re.search(r"zcmb\s*=\s*([\d.]+)", text)

    # Extract distance modulus and uncertainties
    dm_match = re.search(
        r"DM\s*=\s*([\d.]+)\s*\+/-\s*([\d.]+)\s*\+/-\s*([\d.]+)",
        text
    )

    # Skip files with missing values
    if not z_match or not dm_match:
        return None

    return {
        "SN": name,
        "z_CMB": float(z_match.group(1)),
        "DM": float(dm_match.group(1)),
        "DM_stat": float(dm_match.group(2)),
        "DM_sys": float(dm_match.group(3))
    }

In [ ]:
# Extract data from all summary files
sn_data = []

for file in summary_files:

    result = extract_sn_data(file)

    if result is not None:
        sn_data.append(result)

# Convert to a pandas DataFrame
df = pd.DataFrame(sn_data)

# Display the results
display(df)

# Distance Calculations

In [ ]:
# Calculate luminosity distance (Mpc)
df["D_L"] = 10**((df["DM"] - 25) / 5)


# Propagate statistical uncertainty
df["D_L_stat"] = (
    np.log(10) / 5 * df["D_L"] * df["DM_stat"]
)

# Propagate systematic uncertainty
df["D_L_sys"] = (
    np.log(10) / 5 * df["D_L"] * df["DM_sys"]
)


display(df)

# Plots
## Hubble Diagram

In [ ]:
# Combined distance modulus uncertainty
df["DM_err"] = np.sqrt(
    df["DM_stat"]**2 + df["DM_sys"]**2
)

display(df[["SN", "z_CMB", "DM", "DM_err"]])

In [ ]:
# Define cosmological model
cosmo = FlatLambdaCDM(
    H0=70,
    Om0=0.3
)

# Generate redshift values for theoretical curve
z_model = np.linspace(
    df["z_CMB"].min(),
    df["z_CMB"].max(),
    300
)



# Calculate theoretical distance modulus
mu_model = cosmo.distmod(z_model).value


fig, ax = plt.subplots(figsize=(8, 5))

# Plot supernova data with uncertainty bars
ax.errorbar(
    df["z_CMB"],
    df["DM"],
    yerr=df["DM_err"],
    fmt="o",
    capsize=3,
    markersize=5,
    color="navy",
    ecolor="grey",
    label="Type Ia Supernovae"
)

# Label individual supernovae
for _, row in df.iterrows():
    ax.annotate(
        row["SN"],
        (row["z_CMB"], row["DM"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )


# Plot theoretical prediction
ax.plot(
    z_model,
    mu_model,
    color="crimson",
    linestyle="--",
    label=r"Flat $\Lambda$CDM ($H_0=70$)"
)



# Axis labels
ax.set_xlabel("CMB-frame Redshift, $z_{CMB}$")
ax.set_ylabel("Distance Modulus, $\\mu$ (mag)")

ax.set_title("Type Ia Supernova Hubble Diagram")


ax.legend()



fig.savefig(Path("../figures/Hubble_Diagram-v1.png"), dpi=200, bbox_inches="tight")
plt.show()

## Hubble Residuals

In [ ]:
# Calculate theoretical distance modulus at each observed redshift
df["DM_model"] = cosmo.distmod(df["z_CMB"].to_numpy()).value

# Calculate Hubble residuals
df["DM_residual"] = df["DM"] - df["DM_model"]

# Display results
display(
    df[["SN", "z_CMB", "DM", "DM_model", "DM_residual", "DM_err"]]
)

fig, ax = plt.subplots(figsize=(8, 5))

ax.errorbar(
    df["z_CMB"],
    df["DM_residual"],
    yerr=df["DM_err"],
    fmt="o",
    capsize=3,
    color="navy",
    ecolor="grey"
)

# Zero residual reference line
ax.axhline(
    0,
    color="crimson",
    linestyle="--",
    label=r"Flat $\Lambda$CDM ($H_0=70$)"
)

# Label each supernova
for _, row in df.iterrows():
    ax.annotate(
        row["SN"],
        (row["z_CMB"], row["DM_residual"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8
    )

ax.set_xlabel(r"CMB-frame Redshift, $z_{\mathrm{CMB}}$")
ax.set_ylabel(r"Hubble Residual, $\Delta\mu$ (mag)")
ax.set_title("Type Ia Supernova Hubble Residuals")


ax.legend()

fig.savefig(Path("../figures/Hubble_Residuals-v1.png"), dpi=200, bbox_inches="tight")
plt.show()